# Studio / 1BR rent model — Santa Cruz & Monterey

**Roadmap.** We'll work through these together, one at a time.

| # | Step | Status |
|---|------|--------|
| 0 | Setup and load | |
| 1 | Look at the target | |
| 2 | Build the train/test split | |
| 3 | Baseline: median by `place` × `bedrooms` | |
| 4 | Ridge on `log_price` | |
| 5 | LightGBM | |
| 6 | Evaluate: MAE / MAPE, broken out by sub-market | |
| 7 | Ablation: drop `zori_zip` + `tract_*` and see what's left | |

Rule for this notebook: **no `sklearn` import until step 3.** Steps 1–2 are
plain pandas. Most modeling mistakes happen before a model is ever fit.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.width", 120)
pd.set_option("display.max_columns", 50)

df = pd.read_parquet("../data/processed/features.parquet")

# The modeling slice: studios and 1-bedrooms in the two target counties.
t = df[df.is_studio_1br & df.county.isin(["Santa Cruz", "Monterey"])].copy()

print(f"all events      {len(df):,}")
print(f"studio/1br      {len(t):,}")
print(f"properties      {t.id.nunique():,}")
print(f"date range      {t.listedDate.min():%Y-%m} .. {t.listedDate.max():%Y-%m}")

In [ ]:
# Distribution of the target
t.price.describe()
# t.price.hist(bins=50)
# t.log_price.hist(bins=50)

# Try: t.price.hist(bins=50)  vs  t.log_price.hist(bins=50)
# Try: t.groupby(t.listedDate.dt.year).price.median()
t.groupby(t.listedDate.dt.year).price.median()

In [ ]:
# Your split goes here.
#
CUTOFF = pd.Timestamp("2026-01-01")
train = t[t.listedDate < CUTOFF]
test  = t[t.listedDate >= CUTOFF]
#
overlap = set(train.id) & set(test.id)

print(f"properties on both sides: {len(overlap)}")
train = train[~train.id.isin(overlap)]


---
Once your split is built and you've handled the overlap, stop and we'll talk
through it before touching a model.

In [ ]:
for c in ["2022-01-01", "2024-01-01", "2025-07-01", "2026-01-01"]:
    C = pd.Timestamp(c)
    n_train = (t.listedDate < C).sum()
    n_test  = (t.listedDate >= C).sum()
    print(f"{c}   train {n_train:>6,}   test {n_test:>6,}   test {n_test/len(t):.0%}")

In [ ]:
t.listedDate.dt.year.value_counts().sort_index()

In [ ]:
t.listedDate.quantile(0.75)

In [ ]:
t["coord"] = t.latitude.round(5).astype(str) + "," + t.longitude.round(5).astype(str)

train = t[t.listedDate < CUTOFF]
test  = t[t.listedDate >= CUTOFF]

overlap = set(train.coord) & set(test.coord)
train = train[~train.coord.isin(overlap)]

In [ ]:
def describe_split(train, test):
    for name, d in [("train", train), ("test", test)]:
        print(f"{name:<6} events {len(d):>6,}  properties {d.id.nunique():>6,}  "
              f"median ${d.price.median():>6,.0f}  "
              f"{d.listedDate.min():%Y-%m} .. {d.listedDate.max():%Y-%m}")

describe_split(train, test)

In [ ]:
overlap = set(train.id) & set(test.id)

print(f"properties on both sides: {len(overlap)}")

In [ ]:
print(train.groupby(train.listedDate.dt.year).price.median())
print(test.groupby(test.listedDate.dt.year).price.median())
print(len(set(train.coord) & set(test.coord)))

In [ ]:
lookup = train.groupby(["place", "bedrooms"])["price"].median()

lookup_df = lookup.reset_index().rename(columns={"price": "baseline"})
test = test.merge(lookup_df, on=["place", "bedrooms"], how="left")

print(f"{len(lookup)} groups in the lookup table")
print(f"test rows with no match: {test['baseline'].isna().sum()}")

In [ ]:
studio_median = train[train.bedrooms == 0].price.median()
onebr_median  = train[train.bedrooms == 1].price.median()

fallback = test.bedrooms.map({0: studio_median, 1: onebr_median})
test["baseline"] = test["baseline"].fillna(fallback)

print(studio_median)
print(onebr_median)
print(test["baseline"].isna().sum())

In [ ]:
err = test["baseline"] - test["price"]

mae  = err.abs().mean()
mape = (err.abs() / test["price"]).mean() * 100
bias = err.mean()

print(f"MAE   ${mae:,.0f}")
print(f"MAPE  {mape:.1f}%")
print(f"bias  ${bias:+,.0f}")

In [ ]:
NUMERIC = [
    "bedrooms", "bathrooms", "squareFootage",
    "dist_coast_mi", "dist_ucsc_mi", "dist_csumb_mi",
    "dist_hwy17_mi", "dist_town_center_mi", "dist_sc_downtown_mi",
    "latitude", "longitude",
    "months_since_2020", "listed_month",
    "zori_zip",
    "tract_median_hh_income", "tract_median_gross_rent",
    "tract_pct_renter", "tract_pop_density_sqmi",
]

#CATEGORICAL = ["place", "propertyType", "place_kind", "county"]
CATEGORICAL = ["propertyType", "place_kind"]

TARGET = "log_price"

# check every column actually exists — catches typos now instead of later
missing = [c for c in NUMERIC + CATEGORICAL + [TARGET] if c not in train.columns]
print("columns not found:", missing)
print(f"{len(NUMERIC)} numeric + {len(CATEGORICAL)} categorical features")

In [ ]:
# Two flags that say "this value was imputed, don't fully trust it"
NUMERIC = NUMERIC + ["sqft_missing", "zori_missing"]

NUMERIC = [c for c in NUMERIC if c not in ("latitude", "longitude")]
NUMERIC = [c for c in NUMERIC
           if c not in ("dist_hwy17_mi", "dist_sc_downtown_mi", "dist_csumb_mi")]

# Learn fill values from TRAIN ONLY
fill_values = train[NUMERIC].median()

X_train = train[NUMERIC].fillna(fill_values)
X_test  = test[NUMERIC].fillna(fill_values)

print("remaining NaN — train:", X_train.isna().sum().sum(),
      " test:", X_test.isna().sum().sum())
print(f"\nfill values used for the gappy columns:")
print(fill_values[["squareFootage", "zori_zip",
                   "tract_median_hh_income", "tract_median_gross_rent"]].to_string())
print(f"\nX_train {X_train.shape}   X_test {X_test.shape}")

In [ ]:
from sklearn.preprocessing import OneHotEncoder

encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
encoder.fit(train[CATEGORICAL])          # learn the categories from TRAIN only

cat_names = encoder.get_feature_names_out(CATEGORICAL)

cat_train = pd.DataFrame(encoder.transform(train[CATEGORICAL]),
                         columns=cat_names, index=train.index)
cat_test  = pd.DataFrame(encoder.transform(test[CATEGORICAL]),
                         columns=cat_names, index=test.index)

X_train_full = pd.concat([X_train, cat_train], axis=1)
X_test_full  = pd.concat([X_test,  cat_test],  axis=1)

y_train = train[TARGET]
y_test  = test[TARGET]

print(f"X_train_full {X_train_full.shape}   X_test_full {X_test_full.shape}")
print(f"y_train {y_train.shape}   y_test {y_test.shape}")

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
scaler.fit(X_train)                      # X_train is the 20 numeric columns

num_train = pd.DataFrame(scaler.transform(X_train),
                         columns=X_train.columns, index=X_train.index)
num_test  = pd.DataFrame(scaler.transform(X_test),
                         columns=X_test.columns,  index=X_test.index)

# one-hot columns join unscaled, staying as clean 0/1
X_train_scaled = pd.concat([num_train, cat_train], axis=1)
X_test_scaled  = pd.concat([num_test,  cat_test],  axis=1)

y_train = train[TARGET]
y_test  = test[TARGET]

print(f"X_train_scaled {X_train_scaled.shape}   X_test_scaled {X_test_scaled.shape}")
print(f"numeric cols   mean≈0, std≈1")
first_cat = cat_train.columns[0]
print(f"one-hot cols   still 0/1: {sorted(X_train_scaled[first_cat].unique())}  (e.g. {first_cat})")

In [ ]:
from sklearn.linear_model import Ridge

ridge = Ridge(alpha=1.0)
ridge.fit(X_train_scaled, y_train)

In [ ]:
pred_log = ridge.predict(X_test_scaled)
pred     = pd.Series(np.exp(pred_log), index=test.index)   # log -> dollars
actual   = test["price"]

err  = pred - actual
mae  = err.abs().mean()
mape = (err.abs() / actual).mean() * 100
bias = err.mean()

print(f"RIDGE      MAE ${mae:>6,.0f}   MAPE {mape:>5.1f}%   bias ${bias:>+7,.0f}")
print(f"baseline   MAE $   367   MAPE  15.2%   bias $   -184")

In [ ]:
coefs = pd.Series(ridge.coef_, index=X_train_scaled.columns)

print("NUMERIC — effect on log-rent per 1 standard deviation:\n")
for name, v in coefs[X_train.columns].sort_values().items():
    print(f"   {name:<26}{v:>+8.4f}   ~{(np.exp(v)-1)*100:>+6.1f}% rent")

places = coefs[[c for c in coefs.index
                if c.startswith("place_") and not c.startswith("place_kind")]]
print("\nCHEAPEST places (vs average):")
for name, v in places.sort_values().head(6).items():
    print(f"   {name:<30}{v:>+8.4f}   ~{(np.exp(v)-1)*100:>+6.1f}%")
print("\nMOST EXPENSIVE places:")
for name, v in places.sort_values().tail(6).items():
    print(f"   {name:<30}{v:>+8.4f}   ~{(np.exp(v)-1)*100:>+6.1f}%")

In [ ]:
import lightgbm as lgb

# Trees don't care about collinearity, so the features we dropped come back.
LGB_NUM = ["bedrooms", "bathrooms", "squareFootage", "dist_coast_mi", "dist_ucsc_mi",
           "dist_csumb_mi", "dist_hwy17_mi", "dist_town_center_mi", "dist_sc_downtown_mi",
           "latitude", "longitude", "months_since_2020", "listed_month", "zori_zip",
           "tract_median_hh_income", "tract_median_gross_rent", "tract_pct_renter",
           "tract_pop_density_sqmi", "sqft_missing", "zori_missing"]
LGB_CAT = ["place", "propertyType", "place_kind", "county"]
F = LGB_NUM + LGB_CAT

def prep(d, cats=None):
    X = d[F].copy()
    for c in LGB_CAT:
        X[c] = X[c].astype("category")
        if cats is not None:
            X[c] = X[c].cat.set_categories(cats[c])
    return X

X_train_lgb = prep(train)
cats = {c: X_train_lgb[c].cat.categories for c in LGB_CAT}   # train's vocabulary
X_test_lgb  = prep(test, cats)

lgbm = lgb.LGBMRegressor(n_estimators=600, learning_rate=0.05, num_leaves=31,
                         min_child_samples=20, random_state=42, verbose=-1)
lgbm.fit(X_train_lgb, y_train)

pred_lgb = pd.Series(np.exp(lgbm.predict(X_test_lgb)), index=test.index)
e = pred_lgb - test["price"]

print(f"LIGHTGBM   MAE ${e.abs().mean():>5,.0f}   MAPE {(e.abs()/test['price']).mean()*100:>4.1f}%   bias ${e.mean():>+6,.0f}")
print(f"ridge      MAE $  316   MAPE 14.0%   bias $   +37")
print(f"baseline   MAE $  367   MAPE 15.2%   bias $  -184")

In [ ]:
lgbm_real = lgb.LGBMRegressor(n_estimators=600, learning_rate=0.05, num_leaves=31,
                              min_child_samples=20, random_state=42, verbose=-1)
lgbm_real.fit(X_train_lgb, train["log_real_price"])

pred_real = np.exp(lgbm_real.predict(X_test_lgb))        # in 2026-01 dollars
pred_lgb2 = pd.Series(pred_real / test["deflator_to_ref"].values, index=test.index)

e2 = pred_lgb2 - test["price"]
print(f"LGBM deflated  MAE ${e2.abs().mean():>5,.0f}   MAPE {(e2.abs()/test['price']).mean()*100:>4.1f}%   bias ${e2.mean():>+6,.0f}")
print(f"LGBM nominal   MAE $  292   MAPE 12.4%   bias $   -72")

In [ ]:
from sklearn.linear_model import LinearRegression

# 1. fit a straight-line time trend on TRAIN only
trend = LinearRegression().fit(train[["months_since_2020"]], train["log_price"])

# 2. remove it -- trees now learn only what's left after time
resid_train = train["log_price"] - trend.predict(train[["months_since_2020"]])

lgbm_resid = lgb.LGBMRegressor(n_estimators=600, learning_rate=0.05, num_leaves=31,
                               min_child_samples=20, random_state=42, verbose=-1)
lgbm_resid.fit(X_train_lgb, resid_train)

# 3. add the trend back -- a straight line extrapolates into 2026 fine
pred_log = trend.predict(test[["months_since_2020"]]) + lgbm_resid.predict(X_test_lgb)
pred_lgb3 = pd.Series(np.exp(pred_log), index=test.index)

e3 = pred_lgb3 - test["price"]
print(f"LGBM + trend   MAE ${e3.abs().mean():>5,.0f}   MAPE {(e3.abs()/test['price']).mean()*100:>4.1f}%   bias ${e3.mean():>+6,.0f}")

In [ ]:
months = np.linspace(40, 100, 120)
row = X_test_lgb.iloc[[0]]
grid = pd.concat([row]*len(months), ignore_index=True)
grid["months_since_2020"] = months
for c in LGB_CAT:
    grid[c] = grid[c].astype("category").cat.set_categories(cats[c])

nominal = np.exp(lgbm.predict(grid))
trended = np.exp(trend.predict(pd.DataFrame({"months_since_2020": months}))
                 + lgbm_resid.predict(grid))

plt.figure(figsize=(8,4))
plt.plot(months, nominal, label="LGBM nominal")
plt.plot(months, trended, label="LGBM + own trend")
plt.axvline(train.months_since_2020.max(), ls="--", c="grey", label="end of training data")
plt.xlabel("months_since_2020"); plt.ylabel("predicted rent"); plt.legend(); plt.show()

In [ ]:
test = test.copy()
test["pred"] = pred_lgb3
test["err"] = test["pred"] - test["price"]
test["ape"] = (test["err"] / test["price"]).abs() * 100

by_place = (test.groupby("place")
    .agg(n=("price", "size"), median_rent=("price", "median"),
         MAE=("err", lambda e: e.abs().mean()),
         MAPE=("ape", "mean"), bias=("err", "mean"))
    .query("n >= 15").sort_values("MAPE"))
print(by_place.round(0).to_string())

test["rent_band"] = pd.qcut(test["price"], 5)
print("\n", test.groupby("rent_band", observed=True)
      .agg(n=("price","size"), MAPE=("ape","mean"), bias=("err","mean"))
      .round(1).to_string())

In [ ]:
QUANTILES = {"lo": 0.10, "mid": 0.50, "hi": 0.90}
resid_train = train["log_price"] - trend.predict(train[["months_since_2020"]])

q_models, q_pred = {}, {}
for name, q in QUANTILES.items():
    q_models[name] = lgb.LGBMRegressor(objective="quantile", alpha=q,
                                       n_estimators=600, learning_rate=0.05,
                                       num_leaves=31, min_child_samples=20,
                                       random_state=42, verbose=-1)
    q_models[name].fit(X_train_lgb, resid_train)
    q_pred[name] = np.exp(trend.predict(test[["months_since_2020"]])
                          + q_models[name].predict(X_test_lgb))

lo, mid, hi = q_pred["lo"], q_pred["mid"], q_pred["hi"]
covered = ((test["price"] >= lo) & (test["price"] <= hi)).mean()

print(f"coverage of the 10-90 interval: {covered:.1%}   (target 80%)")
print(f"median interval width: ${np.median(hi - lo):,.0f}")
e = mid - test["price"]
print(f"median model:  MAE ${e.abs().mean():,.0f}   MAPE {(e.abs()/test['price']).mean()*100:.1f}%")

In [ ]:
# ---- 1. carve the most recent 20% of train off as a calibration set ----
cut   = train["months_since_2020"].quantile(0.80)
fit_d = train[train["months_since_2020"] <  cut]
cal_d = train[train["months_since_2020"] >= cut]

# ---- 2. fit a throwaway model that has NEVER seen the calibration listings ----
X_fit, X_cal = prep(fit_d, cats), prep(cal_d, cats)
trend_tmp = LinearRegression().fit(fit_d[["months_since_2020"]], fit_d["log_price"])
tmp = lgb.LGBMRegressor(n_estimators=600, learning_rate=0.05, num_leaves=31,
                        min_child_samples=20, random_state=42, verbose=-1)
tmp.fit(X_fit, fit_d["log_price"] - trend_tmp.predict(fit_d[["months_since_2020"]]))

# ---- 3. measure how wrong it actually is on those held-out listings ----
cal_pred = trend_tmp.predict(cal_d[["months_since_2020"]]) + tmp.predict(X_cal)
cal_err  = np.abs(cal_d["log_price"] - cal_pred)
k = np.quantile(cal_err, 0.80)          # the honest half-width

# ---- 4. apply it to the FULL model, which trained on all of train ----
pred_log = trend.predict(test[["months_since_2020"]]) + lgbm_resid.predict(X_test_lgb)
mid = np.exp(pred_log)
lo  = np.exp(pred_log - k)
hi  = np.exp(pred_log + k)

covered = ((test["price"] >= lo) & (test["price"] <= hi)).mean()
e = mid - test["price"]
print(f"half-width k = {k:.3f} in log space  ->  x/÷ {np.exp(k):.2f}")
print(f"coverage:  {covered:.1%}   (target 80%)")
print(f"median interval width: ${np.median(hi - lo):,.0f}")
print(f"median model:  MAE ${e.abs().mean():,.0f}   MAPE {(e.abs()/test['price']).mean()*100:.1f}%")

test = test.copy()
test["lo"], test["mid"], test["hi"] = lo, mid, hi

In [ ]:
ZORI  = ["zori_zip", "zori_missing"]
TRACT = ["tract_median_hh_income", "tract_median_gross_rent",
         "tract_pct_renter", "tract_pop_density_sqmi"]

def ablate(drop_num=(), drop_cat=(), label=""):
    num = [c for c in LGB_NUM if c not in drop_num]
    cat = [c for c in LGB_CAT if c not in drop_cat]
    Fa = num + cat
    def prep_a(d, c=None):
        X = d[Fa].copy()
        for col in cat:
            X[col] = X[col].astype("category")
            if c is not None: X[col] = X[col].cat.set_categories(c[col])
        return X
    Xa = prep_a(train); ca = {c: Xa[c].cat.categories for c in cat}
    p = np.exp(trend.predict(test[["months_since_2020"]])
               + lgb.LGBMRegressor(n_estimators=600, learning_rate=0.05, num_leaves=31,
                                   min_child_samples=20, random_state=42, verbose=-1)
                 .fit(Xa, resid_train).predict(prep_a(test, ca)))
    e = p - test["price"]
    print(f"   {label:<40}{len(Fa):>4}{e.abs().mean():>9,.0f}"
          f"{(e.abs()/test['price']).mean()*100:>8.1f}%")

print(f"   {'model':<40}{'feat':>4}{'MAE':>9}{'MAPE':>9}")
ablate(label="FULL model")
ablate(ZORI, label="without zori_zip")
ablate(TRACT, label="without tract_*")
ablate(ZORI + TRACT, label="without BOTH")
ablate(drop_cat=["place"], label="without place")
ablate(("squareFootage", "sqft_missing"), label="without squareFootage")
ablate(("months_since_2020",), label="without months_since_2020")